In [0]:
dbutils.widgets.text("catalog","fk_1392")
catalog = dbutils.widgets.get("catalog")

In [0]:
# gold.dim_customer
# customer_Sk BIGINT, plus all columns of silver.customers, customer_sk is unique and newver null

spark.sql(f"""
  CREATE OR REPLACE TABLE {catalog}.gold.dim_customer AS
  SELECT 
    customer_id,
    full_name,
    email,
    phone,
    city,
    tier,
    signup_date,
    customer_id AS customer_sk
  FROM {catalog}.silver.customers
          
""")

In [0]:
# gold.dim_store

spark.sql(f"""
  CREATE OR REPLACE TABLE {catalog}.gold.dim_store AS
  SELECT 
    store_id,
    store_name,
    city,
    state,
    region,
    opening_date      
  FROM {catalog}.silver.stores
          
""")

In [0]:
#gold.dim_product

spark.sql(f"""
  CREATE OR REPLACE TABLE {catalog}.gold.dim_product AS
  SELECT 
    product_id,
    product_name,
    brand,
    category_l1,
    category_l2,
    pack_size,
    pack_unit,
    mrp,
    selling_price,
    is_active
    FROM {catalog}.silver.products
""")

In [0]:
#gold.dim_date
#date_key --> int(yyyyMMdd)
#calendar_date --> date
#day_name --> string
#iso_week int ISO 8601 week number
#IS_WEEKEND BOOLEAN sat or sunday

spark.sql(f"""
  CREATE OR REPLACE TABLE {catalog}.gold.dim_date AS
  SELECT
    date_format(signup_date, 'yyyyMMdd') AS date_key,
    signup_date AS calendar_date,
    date_format(calendar_date, 'EEEE') AS day_name,
    EXTRACT(WEEK FROM calendar_date) AS iso_week,
    CASE WHEN day_name IN ('Saturday', 'Sunday') THEN True ELSE False END AS is_weekend
  FROM
    {catalog}.silver.customers
""")

#facts

In [0]:
#gold.fact_orders

#order_date_key should match with dim_date.date_key
spark.sql(f"""
  CREATE OR REPLACE TABLE {catalog}.gold.fact_orders AS
  SELECT     
  order_id AS order_id,
  date_format(order_ts, 'yyyyMMdd') AS order_date_key,
  customer_id AS customer_sk,
  store_id,
  order_status,
  payment_method,
  size(items) AS line_count,
  aggregate(items, 0L, (acc, x) -> acc + x.qty) AS units,
  aggregate(items, CAST(0.0 AS DOUBLE), (acc, x) -> acc + (x.unit_price * x.qty)) AS gross_amount,
  aggregate(items, CAST(0.0 AS DOUBLE), (acc, x) -> acc + x.discount) AS discount_amount,
  aggregate(items, CAST(0.0 AS DOUBLE), (acc, x) -> acc + ((x.unit_price * x.qty) - x.discount)) AS net_item_amount,
  delivery.delivery_fee,
  aggregate(items, CAST(0.0 AS DOUBLE), (acc, x) -> acc + ((x.unit_price * x.qty) - x.discount)) + delivery.delivery_fee AS order_total,
  delivery.promised_mins,
  delivery.actual_mins,
  CASE WHEN delivery.actual_mins > delivery.promised_mins THEN True ELSE False END AS is_on_time
  FROM
  {catalog}.silver.orders
""")

#kpis


In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.agg_daily_store_sales AS
SELECT
  dd.calendar_date AS order_date,
  fo.store_id,
  ds.city,
  SUM(CASE WHEN fo.order_status = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_orders,
  SUM(CASE WHEN fo.order_status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_orders,
  SUM(fo.units) AS units_sold,
  SUM(fo.gross_amount) AS gross_amount
FROM
  {catalog}.gold.fact_orders fo
  JOIN {catalog}.gold.dim_date dd ON fo.order_date_key = dd.date_key
  JOIN {catalog}.gold.dim_store ds ON fo.store_id = ds.store_id
GROUP BY
  dd.calendar_date, fo.store_id, ds.city
""")


In [0]:
#count deliverd and cancelled order , NET_ITEM_REVENUE,

spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.v_kpi_summary AS 
SELECT 
  COUNT(*) AS total_orders,
  SUM(CASE WHEN order_status = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_orders,
  SUM(CASE WHEN order_status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_orders,
  SUM(CASE WHEN order_status = 'DELIVERED' THEN items[*].net_item_revenue ELSE 0 END) AS net_item_revenue

FROM
  {catalog}.silver.orders
""")